<a href="https://colab.research.google.com/github/OutForMilks/Hunger/blob/main/g2p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

**Google Collab**

In [6]:
!git clone https://github.com/OutForMilks/Hunger.git
%cd /content/Hunger

Cloning into 'Hunger'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 32 (delta 4), reused 21 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 35.52 KiB | 17.76 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Hunger


In [7]:
import torch
import torch.nn as nn

from pathlib import Path

from src.prep_data import *
from src.decode import *
from src.train import *

# **Prepare Data**

In [8]:
input_dir = Path("./data")
output_dir = Path("./prepare")
languages = ["fil"]
dataset_splits = ["train", "dev", "text"]

print("Languages:", " ".join(languages))
for split in dataset_splits:
    print(f"[{split}]")
    convert_split(input_dir, output_dir, languages, split)

Languages: fil
[train]
  [skip] data/fil_train.tsv not found
  -> wrote 0 lines to train.{src,tgt,lang}
[dev]
  [skip] data/fil_dev.tsv not found
  -> wrote 0 lines to dev.{src,tgt,lang}
[text]
  [skip] data/fil_text.tsv not found
  -> wrote 0 lines to text.{src,tgt,lang}


# **Train**

In [9]:
DATA = Path("./prepare")
OUTPUT = Path("./models")
#seed is a var
STEPS = 200000
SAVE_STEP = 50000

BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

#loging is a var

CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 1000

cpu


**SEED 1**

In [11]:

seed = 1
config_1 = CONFIG
config_1["seed"] = seed

torch.manual_seed(seed)
os.makedirs(OUTPUT, exist_ok=True)

# Vocabularies are built once from the training data and saved alongside the
# checkpoints so decoding uses the exact same mapping.
src_vocab, tgt_vocab = build_vocabs(
    os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))
print(f"src vocab={len(src_vocab)}  tgt vocab={len(tgt_vocab)}")

train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                        os.path.join(DATA, "train.tgt"),
                        src_vocab, tgt_vocab)
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=collate, drop_last=True)
print(f"training examples: {len(train_ds)}")

model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                    n_heads=NUM_HEADS, d_ff=FF_SIZE,
                    n_layers=N_LAYERS, dropout=DROPOUT,
                    pad_id=PAD_ID).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
sched = NoamLR(opt, HIDDEN_SIZE, WARMUP)
criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING)

model.train()
data_iter = infinite_loader(loader)
running, count = 0.0, 0
for step in range(1, STEPS + 1):
    src, tgt_in, tgt_out, src_pad, tgt_pad = next(data_iter)
    src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
    src_pad, tgt_pad = src_pad.to(DEVICE), tgt_pad.to(DEVICE)

    logits = model(src, tgt_in, src_pad, tgt_pad)
    loss = criterion(logits, tgt_out)

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    sched.step()

    running += loss.item()
    count += 1
    if step % log_step == 0:
        print(f"step {step:>7}/{STEPS}  loss {running / count:.4f}  "
                f"lr {opt.param_groups[0]['lr']:.2e}")
        running, count = 0.0, 0

    if step % SAVE_STEP == 0 or step == STEPS:
        ckpt = os.path.join(OUTPUT, f"ckpt_{step}.pt")
        torch.save({"model": model.state_dict(),
                    "config": config_1,
                    "src_vocab": src_vocab.to_dict(),
                    "tgt_vocab": tgt_vocab.to_dict()}, ckpt)
        print(f"  saved {ckpt}")

src vocab=4  tgt vocab=4


ValueError: num_samples should be a positive integer value, but got num_samples=0

# **Decode**

# **Evaluate**